
# Работа с данными бизнеса в PySpark

### Цели и задачи.       
Руководство сервиса хочет лучше понимать поведение пользователей: какие типы контента они выбирают, как долго его слушают или читают, а также в какие дни недели и через какие платформы (мобильное приложение, веб-версия) это происходит. Эти инсайты позволят улучшить систему рекомендаций и принимать стратегические решения по развитию продукта. Для этого вам понадобится обработать и проанализировать реальные пользовательские данные с помощью PySpark, построить агрегаты и сделать бизнес-выводы на их основе.     
Нужно проанализировать, как различается суммарное время потребления контента в выходные и будние дни, а также выяснить, для какого типа контента наблюдается такая разница — для взрослого или невзрослого.     


#### Описание данных

Таблица `bookmate.audition` содержит данные об активности пользователей и включает столбцы:    
- `audition_id` — уникальный идентификатор сессии чтения или прослушивания;    
- `puid` — идентификатор пользователя;    
- `usage_platform_ru` — название платформы, с помощью которой пользователь взаимодействует с контентом;    
- `msk_business_dt_str` — дата и время события (строка, часовой пояс — МСК);    
- `app_version` — версия приложения;    
- `adult_content_flg` — значение, которое показывает, был ли контент для взрослых ( True или False );    
- `hours` — длительность сессии чтения или прослушивания в часах;    
- `hours_sessions_long` — длительность длинных сессий в часах;    
- `kids_content_flg` — значение, которое показывает, был ли это детский контент ( True или False );     
- `main_content_id` — идентификатор основного контента;    
- `usage_geo_id` — идентификатор географического местоположения пользователя. 

Таблица `bookmate.content` включает столбцы:    
- `main_content_id` — идентификатор основного контента;    
- `main_author_id` — идентификатор основного автора контента;    
- `main_content_type` — тип контента: аудио, текст или другой;    
- `main_content_name` — название контента;    
- `main_content_duration_hours` — длительность контента в часах;     
- `published_topic_title_list` — список жанров или тем контента.     

## Шаг 1. Загрузка данных и знакомство с ними

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, DateType
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

In [ ]:
# Создаём Spark-сессию
spark = SparkSession.builder \
    .appName("*") \
    .config("*", "*") \
    .getOrCreate()

In [3]:
audition_df = (spark.read.
        option("header", False)
        .option("inferSchema", True)
        .csv("audition.csv")
)

content_df = (spark.read.
        option("header", False)
        .option("inferSchema", True)
        .csv("content.csv")
)

In [4]:
print('audition:')
audition_df.printSchema()

print('content:')
content_df.printSchema()

audition:
root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: boolean (nullable = true)
 |-- _c7: double (nullable = true)
 |-- _c8: double (nullable = true)
 |-- _c9: boolean (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)

content:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: double (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)



In [5]:
print('audition:')
audition_df.show(10)

print('content:')
content_df.show(10)

audition:


+-----+---+--------------------+---------------+----------+----------+-----+------------------+------------------+-----+--------+--------------------+---------+
|  _c0|_c1|                 _c2|            _c3|       _c4|       _c5|  _c6|               _c7|               _c8|  _c9|    _c10|                _c11|     _c12|
+-----+---+--------------------+---------------+----------+----------+-----+------------------+------------------+-----+--------+--------------------+---------+
|  162|  0|68296628-f9d6-11e...|        Станция|2024-11-26|      null|false|0.0377777777777777|0.0377777777777777| true|oCURrBKV|              Алматы|Казахстан|
|  213|  1|682966dc-f9d6-11e...|        Станция|2024-11-26|      null|false| 8.333333333333E-4|               0.0| true|qOL0JJL5|              Москва|   Россия|
|   63|  2|682966dc-f9d6-11e...|        Станция|2024-11-26|      null|false|0.0044444444444444|               0.0| true|ndM5nzgT|             Иркутск|   Россия|
|    2|  4|68296704-f9d6-11e...|  

+--------+---------+--------------------+----------+--------------------+--------------------+
|     _c0|      _c1|                 _c2|       _c3|                 _c4|                 _c5|
+--------+---------+--------------------+----------+--------------------+--------------------+
|A00hxVIL|     Book|Давай поговорим о...|     4.151|'Синхронизировано...|        Карл Ричардс|
|A03if3j5|Comicbook|Минимализм из ком...| 2.6854546|'Уборка и организ...|Элизабет Энрайт Ф...|
|A05XPJgr|     Book| Гоните ваши денежки| 6.9114366|'Художественная л...|Наталья Александрова|
|A06fBP22|     Book|Крайон. Судьбу мо...| 3.5218182|'Эзотерика', 'Окк...|        Тамара Шмидт|
|A0FkgwIl|Comicbook|Майор Гром. Допол...|0.20363636|           'Комиксы'|    Артём Габрелянов|
|A0OKAjnf|Audiobook|Радикальное Проще...| 5.0780554|'Психология', 'Са...|       Колин Типпинг|
|A0e7vsJT|     Book|Обоняние. Увлекат...|  8.896182|'Наука', 'Психоло...|        Паоло Пелоси|
|A0fAe01q|Audiobook|       Амулет ведьмы| 10.47333

### Промежуточный вывод:    
Столбцы не имеют названий, необходимо проименовать. Для столбцов, содержащих идентификаторы, такие как  audition_id, puid, main_content_id, main_author_id, возможно, стоит запретить наличие null-строк.


In [6]:
# подготовим списки новых имен
# в данных есть новые столбцы, которых нет в описании - предположительно, один столбец
# индекс, второй страна
audition_cols = [
    'index',
    'audition_id',
    'puid',
    'usage_platfrom_ru',
    'msk_business_dt_str',
    'app_version',
    'adult_content_flg',
    'hours',
    'hours_sessions_long',
    'kids_content_flg',
    'main_content_id',
    'usage_geo_id',
    'country'
]

content_cols = [
    'main_content_id',
    'main_content_type',
    'main_content_name',
    'main_content_duration_hours',
    'published_topic_title_list',
    'main_author_id'
]

In [7]:
audition_df = audition_df.toDF(*audition_cols)
content_df = content_df.toDF(*content_cols)

print('audition:')
audition_df.show(10)

print('content:')
content_df.show(10)

audition:


+-----+-----------+--------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+--------------------+---------+
|index|audition_id|                puid|usage_platfrom_ru|msk_business_dt_str|app_version|adult_content_flg|             hours|hours_sessions_long|kids_content_flg|main_content_id|        usage_geo_id|  country|
+-----+-----------+--------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+--------------------+---------+
|  162|          0|68296628-f9d6-11e...|          Станция|         2024-11-26|       null|            false|0.0377777777777777| 0.0377777777777777|            true|       oCURrBKV|              Алматы|Казахстан|
|  213|          1|682966dc-f9d6-11e...|          Станция|         2024-11-26|       null|            false| 8.333333333333E-4|                0.0|     

In [8]:
audition_df = audition_df.withColumn(
    'msk_business_dt_str',
    audition_df.msk_business_dt_str.cast(DateType())
)

audition_df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- audition_id: integer (nullable = true)
 |-- puid: string (nullable = true)
 |-- usage_platfrom_ru: string (nullable = true)
 |-- msk_business_dt_str: date (nullable = true)
 |-- app_version: string (nullable = true)
 |-- adult_content_flg: boolean (nullable = true)
 |-- hours: double (nullable = true)
 |-- hours_sessions_long: double (nullable = true)
 |-- kids_content_flg: boolean (nullable = true)
 |-- main_content_id: string (nullable = true)
 |-- usage_geo_id: string (nullable = true)
 |-- country: string (nullable = true)



In [9]:
# кэшируем итоговые датафреймы для ускорения расчетов
audition_df.cache()
content_df.cache()

DataFrame[main_content_id: string, main_content_type: string, main_content_name: string, main_content_duration_hours: double, published_topic_title_list: string, main_author_id: string]

In [10]:
print(f'Количество строк audition_df: {audition_df.count()}')

Количество строк audition_df: 1002896


In [11]:
print(f'Количество строк content_df: {content_df.count()}')

Количество строк content_df: 31668


### Промежуточный вывод:    
Данные загружены, предварительное изучение показало, что таблица `audition_df` отличается от описания данных, наличием двух дополнительных столбцов. Предположительно, первый столбец это индекс, оставшийся после выгрузки таблицы. Последний столбец обозначает страну, которой принадлежит город геопозиции пользователя.    
Во второй таблице лишних столбцов нет.    
Имена таблицы не указаны, поэтому было проведено ручное переименование.   
Для поля `msk_business_dt_str` изменен тип данных на DateType.     
Таблица `audition_df` содержит 1002896 строк, таблица `content_df` 31668. Такая разница обусловлена тем, что первая таблица содержит данные сессий (продолжительность, частота, геопозиция, дата и время, вид контента, и т.п), а вторая лишь данные по контенту - книгам (наименование, автор, теги и т.д.).

## Шаг 2. Трансформация и преобразование таблиц

In [12]:
audition_df_agg = (audition_df
    .withColumn('day_of_week', (F.dayofweek('msk_business_dt_str') + 5) % 7 + 1)
    .withColumn('is_weekend', 
                F.when(F.col('day_of_week').between(1, 5), F.lit(False))
                 .when(F.col('day_of_week').between(6, 7), F.lit(True)))
    .withColumn('minutes_sessions_long', (F.col('hours_sessions_long') * 60)
                .cast(IntegerType()))
)

audition_df_agg.cache()
audition_df_agg.show(10)

+-----+-----------+--------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+--------------------+---------+-----------+----------+---------------------+
|index|audition_id|                puid|usage_platfrom_ru|msk_business_dt_str|app_version|adult_content_flg|             hours|hours_sessions_long|kids_content_flg|main_content_id|        usage_geo_id|  country|day_of_week|is_weekend|minutes_sessions_long|
+-----+-----------+--------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+--------------------+---------+-----------+----------+---------------------+
|  162|          0|68296628-f9d6-11e...|          Станция|         2024-11-26|       null|            false|0.0377777777777777| 0.0377777777777777|            true|       oCURrBKV|              Алматы|Казахстан|          2|     f

In [13]:
minutes_df = audition_df_agg.select(
    'puid', 
    'hours_sessions_long', 
    'minutes_sessions_long', 
    'is_weekend',
    'adult_content_flg'
)

minutes_group = minutes_df.groupBy(['is_weekend', 'adult_content_flg']).agg(
    F.sum('minutes_sessions_long').alias('total_minutes')
).orderBy(['adult_content_flg', 'is_weekend'])

minutes_group.show()

+----------+-----------------+-------------+
|is_weekend|adult_content_flg|total_minutes|
+----------+-----------------+-------------+
|     false|            false|      3369978|
|      true|            false|      1401035|
|     false|             true|     14625175|
|      true|             true|      5197958|
+----------+-----------------+-------------+



In [14]:
# функция для выгрузки кэшированных датафреймов
def clear_cache(df: DataFrame) -> None:
    if df.is_cached:
        df.unpersist()
        print('done')
    else:
        print('df is not cached')

In [15]:
clear_cache(audition_df_agg)

done


### Промежуточный вывод:     
Самые высокие показатели времени у пары `Рабочий день - Взрослый контент` - 14 625 175 мин.   
На втором месте `Выходной день - Взрослый контент` - 5 197 958 мин.    
На последнем месте `Выходной день - Детский контент` - 1 401 035 мин.    
В будние дни взрослый контент занимает в 3 раза больше времени по количеству потребления, чем в выходные. Очевидно, взрослые пользователи пользуются книгами по дороге на работу, или в процессе работы.   
Детский контент в будние дни так же в 2 раза выше, чем в выходные дни.    

## Шаг 3. Соединение таблиц

In [16]:
merged_df = audition_df.join(
    content_df,
    on = 'main_content_id',
    how = 'left'
)

merged_df.count()

1002896

In [17]:
merged_clean_df = merged_df.drop('main_author_id', 'app_version', 'usage_geo_id')

merged_clean_df.cache()
merged_clean_df.show(10)

+---------------+-----+-----------+--------------------+-----------------+-------------------+-----------------+------------------+-------------------+----------------+---------+-----------------+--------------------+---------------------------+--------------------------+
|main_content_id|index|audition_id|                puid|usage_platfrom_ru|msk_business_dt_str|adult_content_flg|             hours|hours_sessions_long|kids_content_flg|  country|main_content_type|   main_content_name|main_content_duration_hours|published_topic_title_list|
+---------------+-----+-----------+--------------------+-----------------+-------------------+-----------------+------------------+-------------------+----------------+---------+-----------------+--------------------+---------------------------+--------------------------+
|       oCURrBKV|  162|          0|68296628-f9d6-11e...|          Станция|         2024-11-26|            false|0.0377777777777777| 0.0377777777777777|            true|Казахстан|   

In [18]:
merged_clean_df.select('puid').distinct().count()

31063

In [19]:
audition_df.select('puid').distinct().count()

31063

In [20]:
unique = [row['main_content_type'] 
          for row in merged_clean_df.select('main_content_type').distinct().collect()]
print(unique)

['Audiobook', None, 'Book', 'Comicbook']


In [21]:
clear_cache(audition_df)
clear_cache(content_df)
clear_cache(merged_clean_df)

done
done
done


In [22]:
spark.stop()

### Промежуточный вывод:     
Количество строк после объединения датафреймов соответствует количеству строк левого датафрейма, содержащего информацию о сессиях и пользователях - 1 002 896.    
Количество уникальных puid в объединенной таблице не отличается от изначального количества.    


## Итоговый вывод:    
Первичный анализ показал, что данные изначально не имеют наименований столбцов. Для столбцов, содержащих идентификаторы, такие как audition_id, puid, main_content_id, main_author_id, возможно, стоит запретить наличие null-строк.   
Первый столбец датафрейма audition возможно, появился после выгрузки и обозначает индекс. Последний столбец обозначает страну для города из usage_geo_id. Описание этих полей отсутствует.     
Во второй таблице лишних столбцов нет.     
Проведено ручное переименование.    
Для поля msk_business_dt_str изменен тип данных на DateType.     
Таблица audition_df содержит 1002896 строк, таблица content_df 31668. Такая разница обусловлена тем, что первая таблица содержит данные сессий (продолжительность, частота, геопозиция, дата и время, вид контента, и т.п), а вторая лишь данные по контенту - книгам (наименование, автор, теги и т.д.).    
Проведен анализ количества потребления времени в минутах для контента.     
Самые высокие показатели времени у пары Рабочий день - Взрослый контент - 14 625 175 мин.
На втором месте Выходной день - Взрослый контент - 5 197 958 мин.    
На последнем месте Выходной день - Детский контент - 1 401 035 мин.    
В будние дни взрослый контент занимает в 3 раза больше времени по количеству потребления, чем в выходные. Очевидно, взрослые пользователи пользуются книгами по дороге на работу, или в процессе работы.     
Детский контент в будние дни так же в 2 раза выше, чем в выходные дни.     
Количество строк после объединения датафреймов соответствует количеству строк левого датафрейма, содержащего информацию о сессиях и пользователях - 1 002 896.    
Количество уникальных puid в объединенной таблице не отличается от изначального количества.  